## Education Data Scrapper

In [1]:
from collections import OrderedDict
from selenium import webdriver
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from datetime import time
import time
import csv


url_list = [
    'https://www.linkedin.com/in/muhammad-aneeq-ashraf-74bb56247/'
    'https://www.linkedin.com/in/shonbutani/',
    'https://www.linkedin.com/in/katherine-poblete-a6a9895b/',
    'https://www.linkedin.com/in/taro-masuda-78a128176/',
    'https://www.linkedin.com/in/xuanzihe/'
]

# Signing In
driver = webdriver.Chrome()
driver.get("https://www.linkedin.com/login")

username = driver.find_element(By.ID, "username")
password = driver.find_element(By.ID, "password")
button = driver.find_element(By.CLASS_NAME, "btn__primary--large")

username.send_keys("malikumairali2004@gmail.com")
password.send_keys("Empire@27")
button.click()

In [14]:
def scroll_down():
    start = time.time()

    initialScroll = 0
    finalScroll = 1000

    while True:
        driver.execute_script(f"window.scrollTo({initialScroll},{finalScroll})")
        initialScroll = finalScroll
        finalScroll += 1000
        
        time.sleep(3)
        
        end = time.time()
        if round(end - start) > 20:
            break

def get_source(profile_url):
    driver.get(profile_url)

    scroll_down()
    
    src = driver.page_source
    soup = BeautifulSoup(src, "html.parser")
    return soup

def scrap_name():
    
    name = driver.find_element(By.CLASS_NAME, "text-heading-xlarge").text
    
    return name


def scrap_education():
    education_list = []
    tabs = driver.find_elements(By.CLASS_NAME, "artdeco-card")
    education_printed = False  # Flag to ensure education details are printed once
    
    for tab in tabs:
        if tab:
            if isinstance(tab.text, str):
                if tab.text:
                    tab_text = tab.text
                    if tab_text.startswith('Education'):
                        done = True
                        if tab_text.endswith('education'):
                            education = tab_text
                            # Split education text by new lines and add to the education_list
                            education_list.extend(education.split('\n'))
                        else:
                            # Split tab_text by new lines and add to the education_list
                            education_list.extend(tab_text.split('\n'))
                            education_printed = True
                    else:
                        continue
                else:
                    continue
            else:
                continue
        else:
            continue
    
    # Remove duplicates and maintain order
    unique_education_dict = OrderedDict.fromkeys(education_list)
    
    if unique_education_dict:
        unique_education_dict.popitem()
    
    education_list = list(unique_education_dict.keys())
    return education_list


def add_education_to_csv(name, profile_url, education_list):
    field_names = ['Name', 'URL', 'Education']
    #try:
    with open('education.csv', mode='a', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=field_names)
            
        file.seek(0, 2)
        file_empty = file.tell() == 0
    
        if file_empty:
            writer.writeheader()

        # Join the education_list elements with '|'
        education_data = ' | '.join(education_list)

        # Write the joined data to the CSV as a single cell
        writer.writerow({'Name': name, 'URL': profile_url, 'Education': education_data})
        print(f"Data Added Successfully!")
            
    #except IOError:
        #print("Data could not be added!")


def main():
    with open('cleanedUrls.csv', mode='r', encoding='utf-8') as file:
        csv_reader = csv.DictReader(file)
        for index, row in enumerate(csv_reader):
            if index < 821:
                continue 
            
            profile_url = row['URLs']
            name = row['Name']
            soup = get_source(profile_url)
            #name = scrap_name()
            time.sleep(12)
            education_list = scrap_education()
            time.sleep(8)
            add_education_to_csv(name, profile_url, education_list)
        
main()


Data Added Successfully!
Data Added Successfully!
Data Added Successfully!
Data Added Successfully!
Data Added Successfully!
Data Added Successfully!
Data Added Successfully!
Data Added Successfully!
Data Added Successfully!
Data Added Successfully!
Data Added Successfully!
Data Added Successfully!
Data Added Successfully!
Data Added Successfully!
Data Added Successfully!
Data Added Successfully!
Data Added Successfully!
Data Added Successfully!
Data Added Successfully!
Data Added Successfully!
Data Added Successfully!
Data Added Successfully!
Data Added Successfully!
Data Added Successfully!
Data Added Successfully!
Data Added Successfully!
Data Added Successfully!
Data Added Successfully!
Data Added Successfully!
Data Added Successfully!
Data Added Successfully!
Data Added Successfully!
Data Added Successfully!
Data Added Successfully!
Data Added Successfully!
Data Added Successfully!
Data Added Successfully!
Data Added Successfully!
Data Added Successfully!
Data Added Successfully!


NoSuchWindowException: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=120.0.6099.72)
Stacktrace:
	GetHandleVerifier [0x00007FF736E44D02+56194]
	(No symbol) [0x00007FF736DB04B2]
	(No symbol) [0x00007FF736C576AA]
	(No symbol) [0x00007FF736C30AFD]
	(No symbol) [0x00007FF736CCC9AB]
	(No symbol) [0x00007FF736CE201F]
	(No symbol) [0x00007FF736CC5C23]
	(No symbol) [0x00007FF736C94A45]
	(No symbol) [0x00007FF736C95AD4]
	GetHandleVerifier [0x00007FF7371BD5BB+3695675]
	GetHandleVerifier [0x00007FF737216197+4059159]
	GetHandleVerifier [0x00007FF73720DF63+4025827]
	GetHandleVerifier [0x00007FF736EDF029+687785]
	(No symbol) [0x00007FF736DBB508]
	(No symbol) [0x00007FF736DB7564]
	(No symbol) [0x00007FF736DB76E9]
	(No symbol) [0x00007FF736DA8094]
	BaseThreadInitThunk [0x00007FF9ACA2257D+29]
	RtlUserThreadStart [0x00007FF9AD56AA58+40]


## Education Data Cleaning

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("education.csv")

In [35]:
import pandas as pd
import numpy as np  # Import numpy for NaN handling
import re

# Read the original CSV file
df = pd.read_csv('education.csv')

def extract_tenure(detail):
    pattern = r'\b2\d{3}\b-\b2\d{3}\b'
    match = re.search(pattern, detail)
    if match:
        return match.group()  # Return the matched year range
    return None

# Function to process and expand rows
def process_education(row):
    if isinstance(row['Education'], str):  # Check if it's a string
        details = row['Education'].split('|')
    else:
        details = [str(row['Education'])]  # Convert non-string values to string
    
    if isinstance(row['Name'], str):  # Check if it's a string
        name = row['Name']
    else:
        name = str(row['Name'])  # Convert non-string values to string

    data = []
    for detail in details:
        detail = detail.strip()
        
        if 'University' in detail:
            university_name = detail
            data.append([name, university_name, None, None])
        
        elif any(keyword in detail for keyword in ['Degree', 'BS ', 'Master', 'Bachelor of Science', 'MBBS', 'PhD', 'Doctor of Philosophy', 'Engineering', 'Major', 'Thesis', 'Pre-Master', 'Bachelor of Technology', 'BE']):
            field_of_study = detail
            if data:
                for item in data:
                    item[2] = field_of_study
        
        # Extract tenure using regex
        tenure = extract_tenure(detail)
        if tenure and data:
            for item in data:
                item[3] = tenure

    return pd.DataFrame(data, columns=['Name', 'University', 'Field_of_Study', 'Tenure'])

# Apply function to each row and concatenate results
processed_data = pd.concat(df.apply(process_education, axis=1).tolist(), ignore_index=True)

# Save the processed data to a new CSV file
processed_data.to_csv('processed_education.csv', index=False)


In [25]:
df2 = pd.read_csv("processed_education.csv")

In [31]:
items = df2["Field_of_Study"].values

In [ ]:
for item in items:
    if ""